In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)

import camelot

from getElement import get_element

In [2]:
# Read table from pdf using code-page number combos

pages = {'3210': '29', '4409': '38', '4410': '40', '3262': '38', '3285': '29', '3239': '42', '3288': '33', '3215': '48'}

list_of_dfs = []
for key, value in pages.items():
    tbl = camelot.read_pdf('./FCT/FCT2017SA.pdf', flavor='stream', pages=value)
    # Convert table to pandas dataframe
    df = tbl[0].df

    # Clean column names
    df = df.add_prefix('col_')

    cols = df.columns.tolist()
    df = df[cols].replace(to_replace='µ', value='micro-', regex=True)

    # Clean up dataframe: remove unnecessary columns, rows
    df = df.loc[3:, :]
        
    # Drop columns
    df.drop(columns=['col_0'], inplace=True)

    # Reset index
    df = df.reset_index(drop=True)

    # Drop empty rows
    # First replace blanks with np.nan
    data_cols = list(df.loc[:, 'col_3':'col_14'])

    df[data_cols] = df[data_cols].replace({'': np.nan})

    # Drop rows
    df.dropna(thresh=12, inplace=True)

    df = df.reset_index(drop=True)

    # Get header data
    df_header = df[(df.index >= 0) & (df.index <= 2)]
    cols = list(df_header)
    #df_header[cols] = df_header[cols].replace(to_replace='(\\n)', value='micro-', regex=True)

    # Select rows of interest based on col_1 == food code
    idx = df[df['col_1'] == key].index[0]

    df = df[(df.index >= idx) & (df.index <= idx+2)]

    # Concat together
    df = pd.concat([df_header, df], axis=0).reset_index(drop=True)

    # Get Code, Description
    df_desr = df.iloc[3,0:2].tolist()

    # Split into individual dfs
    df_macro = df.loc[:, 'col_3':'col_7']
    df_mineral = df.loc[:, 'col_8':'col_10']
    df_vitamin = df.loc[:, 'col_11':'col_14']

    # Pull out header
    df_macro_header = df_macro.head(3).to_numpy().flatten().tolist()
    df_mineral_header = df_mineral.head(3).to_numpy().flatten().tolist()
    df_vitamin_header = df_vitamin.head(3).to_numpy().flatten().tolist()

    # Pull out values
    df_macro_values = df_macro.loc[3:, :].to_numpy().flatten().tolist()
    df_mineral_values = df_mineral.loc[3:, :].to_numpy().flatten().tolist()
    df_vitamin_values = df_vitamin.loc[3:, :].to_numpy().flatten().tolist()

    # Concat them together

    # Dict of Zipped values
    df_macro_values = dict(zip(df_macro_header, df_macro_values))
    df_mineral_values = dict(zip(df_mineral_header, df_mineral_values))
    df_vitamin_values = dict(zip(df_vitamin_header, df_vitamin_values))

    # Create dataframe
    df_macro = pd.DataFrame(df_macro_values, index=[0])
    df_mineral = pd.DataFrame(df_mineral_values, index=[0])
    df_vitamin = pd.DataFrame(df_vitamin_values, index=[0])

    # Concat
    df = pd.concat([df_macro, df_mineral, df_vitamin], axis=1)

    # Add description
    df['Code'] = df_desr[0]
    df['Food Name'] = df_desr[1]

    # Rearrange col order
    cols = df.columns.tolist()
    col_order = cols[-2:] + cols[:-2]

    df = df[col_order]

    list_of_dfs.append(df)
    print(f'Check 3: Everything worked for code # {key}')

output0 = pd.concat(list_of_dfs)
output0

Check 3: Everything worked for code # 3210
Check 3: Everything worked for code # 4409
Check 3: Everything worked for code # 4410
Check 3: Everything worked for code # 3262
Check 3: Everything worked for code # 3285
Check 3: Everything worked for code # 3239
Check 3: Everything worked for code # 3288
Check 3: Everything worked for code # 3215


,Code,Food Name,Mois-g,En-kJ,TotN-g,Prot-g,PlPr-g,AnPr-g,Fat-g,SFA-g,MFA-g,PFA-g,Chol-mg,CHO-g,TFib-g,AdSu-g,Ash-g,Ca-mg,Fe-mg,Mg-mg,P-mg,K-mg,Na-mg,Zn-mg,Cu-mg,Mn-micro-g,A-micro-gRE,Thia-mg,Ribo-mg,Niac-mg,B6-mg,Fol-micro-g,B12-micro-g,Pant-mg,Biot-micro-g,C-mg,D-micro-g,E-mg
0,3210,"Bread/Rolls, white (fortified)",39.0,1036,1.41,8.8,8.8,0.0,1.4,0.24,0.47,0.22,0,45.9,3.2,tr,1.7,16,3.6,43,96,214,653,2.15,0.15,76,109,0.30,0.09,6.0,1.730,80,0.0,0.33,2.3,0,0.00,0.06
0,4409,"Maize meal, sifted, raw",11.8,1365,1.35,8.4,8.4,0.0,3.2,0.42,0.83,1.38,0,69.0,6.7,0.0,0.9,12,3.2,98,90,197,4,2.98,0.16,330,196,0.68,0.23,4.6,0.750,225,0.0,0.47,4.2,tr,0.00,0.62
0,4410,"Maize meal, super, porridge, soft",84.6,266,0.21,1.3,1.3,0.0,0.3,0.04,0.06,0.09,0,13.0,0.7,0.0,0.1,1,0.5,7,11,17,1,0.36,0.02,33,29,0.06,0.03,0.6,0.099,26,0.0,0.06,0.7,0,0.00,0.06
0,3262,"Macaroni/Spaghetti, cooked",66.0,589,-,4.8,4.8,0.0,0.7,0.10,0.08,0.27,0,26.7,1.6,0.0,0.2,7,0.5,18,54,31,1,0.53,0.10,290,0,0.02,0.02,0.4,0.035,7,0.0,0.11,tr,0,0.00,tr
0,3285,"Bread/Rolls, white, high protein",33.2,1146,-,11.7,11.7,0.0,2.4,-,-,-,-,47.8,2.7,0.0,-,82,1.6,29,107,154,513,1.20,0.20,-,-,0.11,0.05,1.1,-,-,-,-,-,-,-,-
0,3239,"Oats, rolled or oatmeal, cooked",85.3,279,-,1.7,1.7,0.0,1.6,0.29,0.51,0.59,0,9.6,1.6,0.0,-,8,0.7,24,77,57,2,0.50,0.06,590,2,0.12,0.02,0.1,0.019,5,0.0,0.20,3.2,0,0.00,0.24
0,3288,"Cake, butter, plain, home-made",24.4,1508,-,5.1,2.5,2.6,12.4,3.22,5.87,2.42,65,55.8,0.8,31.8,-,36,0.8,9,74,81,238,0.44,0.07,200,106,0.06,0.10,0.2,0.025,13,0.4,0.47,4.0,0,1.80,2.41
0,3215,"Rusk, home-made, buttermilk, white (HM)",5.0,1993,-,7.2,5.2,2.0,21.3,5.00,10.51,4.49,34,62.3,1.4,22.9,-,212,1.0,13,352,114,1124,0.53,0.07,530,187,0.07,0.09,0.4,0.038,28,0.2,0.37,2.8,0,1.84,3.89


In [3]:
# FOR PAGE 54  -- !!

# Read table from pdf using code-page number combos

pages = {'3273': '54', '3257': '54', '4416': '54'}

list_of_dfs = []
for key, value in pages.items():
    tbl = camelot.read_pdf('./FCT/FCT2017SA.pdf', flavor='stream', pages=value)
    # Convert table to pandas dataframe
    df = tbl[0].df

    # Clean column names
    df = df.add_prefix('col_')

    df['col_1'] = df['col_1'].str.strip().str.replace('\n', '\\').str.split('\\')
    df = df.reset_index(drop=True)

    df['col_1A'] = np.where(df.index < 4, df['col_1'].apply(get_element, position=0).replace({np.nan: ""}),
        df['col_1'].apply(get_element, position=-1).replace({np.nan: ""}))
    df['col_1B'] = np.where(df.index < 4, df['col_1'].apply(get_element, position=1).replace({np.nan: ""}),
        df['col_1'].apply(get_element, position=0).replace({np.nan: ""}))

    # Reorder and rename cols
    col_names = df.columns.tolist()
    col_order = col_names[:2] + col_names[-2:] + col_names[2:14]
    df = df[col_order]

    df.drop(columns=['col_1'], inplace=True)

    old_names = df.columns.tolist()
    new_names = ['col_0', 'col_1', 'col_2', 'col_3', 'col_4', 'col_5', 'col_6', 'col_7', 'col_8',
    'col_9', 'col_10', 'col_11', 'col_12', 'col_13', 'col_14']

    col_dict = dict(zip(old_names, new_names))

    df = df.rename(columns = col_dict)

    # # Clean up dataframe: remove unnecessary columns, rows
    if key in ['3486']:
        df = df.loc[2:, :]
    else:
        df = df.loc[3:, :]

    # Drop columns
    df.drop(columns=['col_0'], inplace=True)

    # Reset index
    df = df.reset_index(drop=True)

    # Drop empty rows
    # First replace blanks with np.nan
    data_cols = list(df.loc[:, 'col_3':'col_14'])

    df[data_cols] = df[data_cols].replace({'': np.nan})

    # Drop rows
    df.dropna(thresh=12, inplace=True)
    df = df.reset_index(drop=True)

    # Get header data
    df_header = df[(df.index >= 0) & (df.index <= 2)]
    cols = list(df_header)
    df_header[cols] = df_header[cols].replace(to_replace='(\\n)', value='micro-', regex=True)

    # Select rows of interest based on col_1 == food code
    idx = df[df['col_1'] == key].index[0]

    df = df[(df.index >= idx) & (df.index <= idx+2)]

    # Concat together
    df = pd.concat([df_header, df], axis=0).reset_index(drop=True)

    # Get Code, Description
    df_desr = df.iloc[3,0:2].tolist()

    # Split into individual dfs
    df_macro = df.loc[:, 'col_3':'col_7']
    df_mineral = df.loc[:, 'col_8':'col_10']
    df_vitamin = df.loc[:, 'col_11':'col_14']

    # Pull out header
    df_macro_header = df_macro.head(3).to_numpy().flatten().tolist()
    df_mineral_header = df_mineral.head(3).to_numpy().flatten().tolist()
    df_vitamin_header = df_vitamin.head(3).to_numpy().flatten().tolist()

    # Pull out values
    df_macro_values = df_macro.loc[3:, :].to_numpy().flatten().tolist()
    df_mineral_values = df_mineral.loc[3:, :].to_numpy().flatten().tolist()
    df_vitamin_values = df_vitamin.loc[3:, :].to_numpy().flatten().tolist()

    # Concat them together

    # Dict of Zipped values
    df_macro_values = dict(zip(df_macro_header, df_macro_values))
    df_mineral_values = dict(zip(df_mineral_header, df_mineral_values))
    df_vitamin_values = dict(zip(df_vitamin_header, df_vitamin_values))

    # Create dataframe
    df_macro = pd.DataFrame(df_macro_values, index=[0])
    df_mineral = pd.DataFrame(df_mineral_values, index=[0])
    df_vitamin = pd.DataFrame(df_vitamin_values, index=[0])

    # Concat
    df = pd.concat([df_macro, df_mineral, df_vitamin], axis=1)

    # Add description
    df['Code'] = df_desr[0]
    df['Food Name'] = df_desr[1]

    # Rearrange col order
    cols = df.columns.tolist()
    col_order = cols[-2:] + cols[:-2]

    df = df[col_order]

    list_of_dfs.append(df)
    print(f'Check 4: Everything worked for code # {key}')

output1 = pd.concat(list_of_dfs)
output1

Check 4: Everything worked for code # 3273
Check 4: Everything worked for code # 3257
Check 4: Everything worked for code # 4416


,Code,Food Name,Mois-g,En-kJ,TotN-g,Prot-g,PlPr-g,AnPr-g,Fat-g,SFA-g,MFA-g,PFA-g,Chol-mg,CHO-g,TFib-g,AdSu-g,Ash-g,Ca-mg,Fe-mg,Mg-mg,P-mg,K-mg,Na-mg,Zn-mg,Cu-mg,Mn-micro-g,A-micro-gRE,Thia-mg,Ribo-mg,Niac-mg,B6-mg,Fol-micro-g,B12-micro-g,Pant-mg,Biot-micro-g,C-mg,D-micro-g,E-mg
0,3273,"Wheat ﬂour, cake ﬂour",12.5,1499,1.44,8.2,8.2,0.0,0.9,0.13,0.07,0.38,0,76.3,1.7,0.0,0.4,14,1.2,16,85,105,2,0.62,0.14,634,0,0.12,0.04,1.3,0.033,26,0.0,0.46,1.0,0,0.00,0.06
0,3257,"Vetkoek, home-made (cake ﬂour, water)",30.9,1524,-,7.2,4.6,2.6,17.7,2.60,3.67,10.12,87,42.4,1.5,0.0,-,16,1.2,10,87,79,27,0.59,0.10,370,14,0.08,0.10,0.4,0.027,20,0.4,0.62,5.1,0,1.67,9.34
0,4416,"Wheat ﬂour, brown bread ﬂour (fortiﬁed)",14.0,1472,1.99,11.4,11.4,0.0,1.3,-,-,-,0,65.5,6.8,0.0,1.0,7,3.4,55,141,222,7,2.11,0.19,133,190,0.82,0.23,8.0,0.552,499,0.0,0.66,4.4,-,-,0.37


In [4]:
# Read table from pdf using code-page number combos

pages = {'3781': '73', '3486': '214', '4130': '86', '3981': '219', '4007': '219'}

list_of_dfs = []
for key, value in pages.items():

    tbl = camelot.read_pdf('./FCT/FCT2017SA.pdf', flavor='stream', pages=value)
    # Convert table to pandas dataframe
    df = tbl[0].df

    # Clean column names
    df = df.add_prefix('col_')

    cols = df.columns.tolist()
    df = df[cols].replace(to_replace='μ', value='micro-', regex=True)

    # Clean up dataframe: remove unnecessary columns, rows
    df = df.loc[2:, :]

    # Drop columns
    df.drop(columns=['col_0'], inplace=True)

    # Reset index
    df = df.reset_index(drop=True)

    val1 = df['col_12'].str.split(' ').apply(get_element,position=0)[0]
    val2 = df['col_12'].str.split(' ').apply(get_element,position=1)[0]

    df.iloc[0,11] = val1
    df.iloc[0,12] = val2

    if key in '3781':
        val3 = df['col_1'].str.split(' ').apply(get_element,position=0)[0]
        val4 = df['col_1'].str.split(' ').apply(get_element,position=1)[0]

        df.iloc[0,0] = val3
        df.iloc[0,1] = val4
    else:
        pass

    # Drop empty rows
    # First replace blanks with np.nan
    data_cols = list(df.loc[:, 'col_3':'col_14'])

    df[data_cols] = df[data_cols].replace({'': np.nan})

    # Drop rows
    df.dropna(thresh=12, inplace=True)

    df = df.reset_index(drop=True)

    # Get header data
    df_header = df[(df.index >= 0) & (df.index <= 2)]
    cols = list(df_header)
    df_header[cols] = df_header[cols].replace(to_replace='(\\n)', value='micro-', regex=True)

    # Select rows of interest based on col_1 == food code
    idx = df[df['col_1'] == key].index[0]

    df = df[(df.index >= idx) & (df.index <= idx+2)]

    # Concat together
    df = pd.concat([df_header, df], axis=0).reset_index(drop=True)

    # Get Code, Description
    df_desr = df.iloc[3,0:2].tolist()

    # Split into individual dfs
    df_macro = df.loc[:, 'col_3':'col_7']
    df_mineral = df.loc[:, 'col_8':'col_10']
    df_vitamin = df.loc[:, 'col_11':'col_14']

    # Pull out header
    df_macro_header = df_macro.head(3).to_numpy().flatten().tolist()
    df_mineral_header = df_mineral.head(3).to_numpy().flatten().tolist()
    df_vitamin_header = df_vitamin.head(3).to_numpy().flatten().tolist()

    df_macro_header = [x.strip().replace(" ", "") for x in df_macro_header]
    df_mineral_header = [x.strip() for x in df_mineral_header]
    df_vitamin_header = [x.strip() for x in df_vitamin_header]

    # Pull out values
    df_macro_values = df_macro.loc[3:, :].to_numpy().flatten().tolist()
    df_mineral_values = df_mineral.loc[3:, :].to_numpy().flatten().tolist()
    df_vitamin_values = df_vitamin.loc[3:, :].to_numpy().flatten().tolist()

    # Concat them together

    # Dict of Zipped values
    df_macro_values = dict(zip(df_macro_header, df_macro_values))
    df_mineral_values = dict(zip(df_mineral_header, df_mineral_values))
    df_vitamin_values = dict(zip(df_vitamin_header, df_vitamin_values))

    # Create dataframe
    df_macro = pd.DataFrame(df_macro_values, index=[0])
    df_mineral = pd.DataFrame(df_mineral_values, index=[0])
    df_vitamin = pd.DataFrame(df_vitamin_values, index=[0])

    # Concat
    df = pd.concat([df_macro, df_mineral, df_vitamin], axis=1)

    # Add description
    df['Code'] = df_desr[0]
    df['Food Name'] = df_desr[1]

    # Rearrange col order
    cols = df.columns.tolist()
    col_order = cols[-2:] + cols[:-2]

    df = df[col_order]

    list_of_dfs.append(df)
    print(f'Check 3: Everything worked for code # {key}')

output2 = pd.concat(list_of_dfs)
output2

Check 3: Everything worked for code # 3781
Check 3: Everything worked for code # 3486
Check 3: Everything worked for code # 4130
Check 3: Everything worked for code # 3981
Check 3: Everything worked for code # 4007


,Code,Food Name,Mois-g,En-kJ,TotN-g,Prot-g,PlPr-g,AnPr-g,Fat-g,SFA-g,MFA-g,PFA-g,Chol-mg,CHO-g,TFib-g,AdSu-g,Ash-g,Ca-mg,Fe-mg,Mg-mg,P-mg,K-mg,Na-mg,Zn-mg,Cu-mg,Mn-micro-g,A-micro-gRE,Thia-mg,Ribo-mg,Niac-mg,B6-mg,Fol-micro-g,B12-micro-g,Pant-mg,Biot-micro-g,C-mg,D-micro-g,E-mg
0,3781,"Average, other vegetables, boiled",86.4,225,0.22,1.4,1.4,0.0,0.2,0.03,0.02,0.06,0,8.2,3.2,0.0,0.6,23,0.8,18,38,226,19,0.29,0.12,226,12,0.06,0.03,0.5,0.085,18,0.0,0.23,4.2,7,0.00,0.21
0,3486,"Vegetable oil (50% sunflower,",0.0,3700,0.00,0.0,0.0,0.0,100.0,14.84,28.20,52.36,0,0.0,0.0,0.0,-,0,0.0,0,0,0,0,0.00,0.00,0,2,0.00,0.00,0.0,0.000,0,0.0,0.00,0.0,0,0.00,39.22
0,4130,"Mealie, sweetcorn, raw",76.0,422,0.51,3.2,3.2,0.0,1.2,0.18,0.35,0.56,0,16.3,2.7,0.0,0.6,2,0.5,37,89,270,15,0.45,0.05,161,28,0.20,0.06,1.7,0.055,46,0.0,0.76,-,7,0.00,0.09
0,3981,"Cold drink, carbonated, average",89.6,175,-,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0,10.3,0.0,10.3,-,4,0.1,1,3,1,7,0.04,0.01,40,0,0.00,0.00,0.0,0.000,0,0.0,0.00,0.0,0,0.00,0.00
0,4007,"Cold drink, Lucozade",81.7,306,-,tr,-,-,0.0,0.00,0.00,0.00,0,18.0,0.0,8.6,-,4,tr,2,4,2,28,tr,0.01,tr,0,tr,tr,tr,tr,tr,0.0,tr,tr,3,0.00,0.00


In [5]:
pages = {'4288': '229'}

list_of_dfs = []
for key, value in pages.items():

    tbl = camelot.read_pdf('./FCT/FCT2017SA.pdf', flavor='stream', pages=value)
    # Convert table to pandas dataframe
    df = tbl[0].df

    # Clean column names
    df = df.add_prefix('col_')

    cols = df.columns.tolist()
    df = df[cols].replace(to_replace='μ', value='micro-', regex=True)

    # Clean up dataframe: remove unnecessary columns, rows
    df = df.loc[2:, :]


    # Drop columns
    df.drop(columns=['col_0'], inplace=True)

    # Reset index
    df = df.reset_index(drop=True)

    df['col_10A'] = df['col_10'].str.replace('\\n', ' ').str.split(' ').apply(get_element,position=0)
    df['col_10B'] = df['col_10'].str.replace('\\n', ' ').str.split(' ').apply(get_element,position=1)

    val1 = df['col_11'].str.split(' ').apply(get_element,position=0)[0]
    val2 = df['col_11'].str.split(' ').apply(get_element,position=1)[0]

    df.iloc[0,10] = val1
    df.iloc[0,11] = val2

    # Reorder and rename cols
    df.drop(columns=['col_10'], inplace=True)
    col_names = df.columns.tolist()
    col_order = col_names[:9] + col_names[-2:] + col_names[9:12]
    df = df[col_order]

    old_names = df.columns.tolist()
    new_names = ['col_1', 'col_2', 'col_3', 'col_4', 'col_5', 'col_6', 'col_7', 'col_8',
    'col_9', 'col_10', 'col_11', 'col_12', 'col_13', 'col_14']

    col_dict = dict(zip(old_names, new_names))

    df = df.rename(columns = col_dict)

    # Drop empty rows
    # First replace blanks with np.nan
    data_cols = list(df.loc[:, 'col_3':'col_14'])

    df[data_cols] = df[data_cols].replace({'': np.nan})

    # Drop rows
    df.dropna(thresh=12, inplace=True)

    df = df.reset_index(drop=True)

    # Get header data
    df_header = df[(df.index >= 0) & (df.index <= 2)]
    cols = list(df_header)
    #df_header[cols] = df_header[cols].replace(to_replace='(\\n)', value='micro-', regex=True)

    # Select rows of interest based on col_1 == food code
    idx = df[df['col_1'] == key].index[0]

    df = df[(df.index >= idx) & (df.index <= idx+2)]

    # Concat together
    df = pd.concat([df_header, df], axis=0).reset_index(drop=True)

    # Get Code, Description
    df_desr = df.iloc[3,0:2].tolist()

    # Split into individual dfs
    df_macro = df.loc[:, 'col_3':'col_7']
    df_mineral = df.loc[:, 'col_8':'col_10']
    df_vitamin = df.loc[:, 'col_11':'col_14']

    # Pull out header
    df_macro_header = df_macro.head(3).to_numpy().flatten().tolist()
    df_mineral_header = df_mineral.head(3).to_numpy().flatten().tolist()
    df_vitamin_header = df_vitamin.head(3).to_numpy().flatten().tolist()

    df_macro_header = [x.strip().replace(" ", "") for x in df_macro_header]
    df_mineral_header = [x.strip() for x in df_mineral_header]
    df_vitamin_header = [x.strip() for x in df_vitamin_header]

    # Pull out values
    df_macro_values = df_macro.loc[3:, :].to_numpy().flatten().tolist()
    df_mineral_values = df_mineral.loc[3:, :].to_numpy().flatten().tolist()
    df_vitamin_values = df_vitamin.loc[3:, :].to_numpy().flatten().tolist()

    # Concat them together

    # Dict of Zipped values
    df_macro_values = dict(zip(df_macro_header, df_macro_values))
    df_mineral_values = dict(zip(df_mineral_header, df_mineral_values))
    df_vitamin_values = dict(zip(df_vitamin_header, df_vitamin_values))

    # Create dataframe
    df_macro = pd.DataFrame(df_macro_values, index=[0])
    df_mineral = pd.DataFrame(df_mineral_values, index=[0])
    df_vitamin = pd.DataFrame(df_vitamin_values, index=[0])

    # Concat
    df = pd.concat([df_macro, df_mineral, df_vitamin], axis=1)

    # Add description
    df['Code'] = df_desr[0]
    df['Food Name'] = df_desr[1]

    # Rearrange col order
    cols = df.columns.tolist()
    col_order = cols[-2:] + cols[:-2]

    df = df[col_order]

    df['Food Name'] = 'Salt, table, iodised'

    list_of_dfs.append(df)
    print(f'Check 3: Everything worked for code # {key}')

output3 = pd.concat(list_of_dfs)
output3

Check 3: Everything worked for code # 4288


,Code,Food Name,Mois-g,En-kJ,TotN-g,Prot-g,PlPr-g,AnPr-g,Fat-g,SFA-g,MFA-g,PFA-g,Chol-mg,CHO-g,TFib-g,AdSu-g,Ash-g,Ca-mg,Fe-mg,Mg-mg,P-mg,K-mg,Na-mg,Zn-mg,Cu-mg,Mn-micro-g,A-micro-gRE,Thia-mg,Ribo-mg,Niac-mg,B6-mg,Fol-micro-g,B12-micro-g,Pant-mg,Biot-micro-g,C-mg,D-micro-g,E-mg
0,4288,"Salt, table, iodised",2.0,0,00.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0,0.0,0.0,0.0,99.8,92,2.0,092,8,0,38850,0.10,0.10,100,0,00.0,00.0,0.0,0.000,0,0.0,0.00,0.0,0,0.00,0.00


In [6]:
key = '4038'
value = '283'

tbl = camelot.read_pdf('./FCT/FCT2017SA.pdf', flavor='stream', pages=value)
# Convert table to pandas dataframe
df = tbl[0].df

# Clean column names
df = df.add_prefix('col_')

cols = df.columns.tolist()
df = df[cols].replace(to_replace='μ', value='micro-', regex=True)

# Drop columns
df.drop(columns=['col_0'], inplace=True)

# Reset index
df = df.reset_index(drop=True)

# Drop empty rows
# First replace blanks with np.nan
data_cols = list(df.loc[:, 'col_3':'col_14'])

df[data_cols] = df[data_cols].replace({'': np.nan})

# Drop rows
df.dropna(thresh=12, inplace=True)
df = df.reset_index(drop=True)

val1 = df['col_12'].str.split(' ').apply(get_element,position=0)[0]
val2 = df['col_12'].str.split(' ').apply(get_element,position=1)[0]

df.iloc[0,11] = val1
df.iloc[0,12] = val2

# Get header data
df_header = df[(df.index >= 0) & (df.index <= 2)]
cols = list(df_header)
df_header[cols] = df_header[cols].replace(to_replace='(\\n)', value='micro-', regex=True)

# Select rows of interest based on col_1 == food code
idx = df[df['col_1'] == key].index[0]

df = df[(df.index >= idx) & (df.index <= idx+2)]

# Concat together
df = pd.concat([df_header, df], axis=0).reset_index(drop=True)

# Reverse transcribe food name
df.iloc[3,1] = df.iloc[3,1][::-1]

# Get Code, Description
df_desr = df.iloc[3,0:2].tolist()

# Split into individual dfs
df_macro = df.loc[:, 'col_3':'col_7']
df_mineral = df.loc[:, 'col_8':'col_10']
df_vitamin = df.loc[:, 'col_11':'col_14']

# Pull out header
df_macro_header = df_macro.head(3).to_numpy().flatten().tolist()
df_mineral_header = df_mineral.head(3).to_numpy().flatten().tolist()
df_vitamin_header = df_vitamin.head(3).to_numpy().flatten().tolist()

# Pull out values
df_macro_values = df_macro.loc[3:, :].to_numpy().flatten().tolist()
df_mineral_values = df_mineral.loc[3:, :].to_numpy().flatten().tolist()
df_vitamin_values = df_vitamin.loc[3:, :].to_numpy().flatten().tolist()

# Concat them together

# Dict of Zipped values
df_macro_values = dict(zip(df_macro_header, df_macro_values))
df_mineral_values = dict(zip(df_mineral_header, df_mineral_values))
df_vitamin_values = dict(zip(df_vitamin_header, df_vitamin_values))

# Create dataframe
df_macro = pd.DataFrame(df_macro_values, index=[0])
df_mineral = pd.DataFrame(df_mineral_values, index=[0])
df_vitamin = pd.DataFrame(df_vitamin_values, index=[0])

df_macro.columns = [x.strip().replace(' ', '') for x in df_macro.columns]

# Concat
df = pd.concat([df_macro, df_mineral, df_vitamin], axis=1)

# Add description
df['Code'] = df_desr[0]
df['Food Name'] = df_desr[1]

# Rearrange col order
cols = df.columns.tolist()
col_order = cols[-2:] + cols[:-2]

df = df[col_order]

print(f'Check 4: Everything worked for code # {key}')

df

Check 4: Everything worked for code # 4038


,Code,Food Name,Mois-g,En-kJ,TotN-g,Prot-g,PlPr-g,AnPr-g,Fat-g,SFA-g,MFA-g,PFA-g,Chol-mg,CHO-g,TFib-g,AdSu-g,Ash-g,Ca-mg,Fe-mg,Mg-mg,P-mg,K-mg,Na-mg,Zn-mg,Cu-mg,Mn-micro-g,A-micro-gRE,Thia-mg,Ribo-mg,Niac-mg,B6-mg,Fol-micro-g,B12-micro-g,Pant-mg,Biot-micro-g,C-mg,D-micro-g,E-mg
0,4038,"Tea, brewed",7.99,5,00.0,0.0,0.0,0.0,tr,tr,tr,tr,0,0.3,0.0,0.0,-,0,rt,3,1,37,3,0.02,0.01,-,0,00.0,10.0,0.0,0.000,5,0.0,-,-,0,0.00,-


In [7]:
sa_fct = pd.concat([output0, output1, output2, output3, df], axis=0)
sa_fct.to_csv('./data/fct/sa_fct.csv', index=False)